# Human & Animal Detection with YOLOv8 (version 2)
This version **fine-tunes a pretrained YOLOv8 model without throwing away what it already knows** about humans and animals.
Version 1 changed the class list to 11 classes, so the model had to learn them again from very few photos. That is why it mixed up dogs, cows and humans.

**How to use**
1. Runtime > Change runtime type > **T4 GPU** > Save
2. Run Cell 1 to Cell 6 one by one (Shift + Enter). The trained model is saved in your Google Drive.
3. On the lab day, run only **Cell 7** and **Cell 8** (Cell 9 and Cell 10 are optional extra tests).

**Quick check without any training:** in Cell 7 use the line `MODEL_PATH = "yolov8s.pt"`, then run Cell 7 and Cell 8.

In [ ]:
# Step 1: Install, check the GPU, connect Google Drive and choose the settings
!pip install -q ultralytics

import json
import random
import shutil
from pathlib import Path

import torch
from google.colab import drive

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run again.")

drive.mount('/content/drive')

# ---------- Settings (you can change these) ----------
BASE_MODEL = "yolov8s.pt"     # pretrained model that already knows humans and animals ("yolov8m.pt" = more accurate but slower)
EPOCHS = 20                   # fine-tuning rounds (the model already knows a lot, so 20 is enough)
IMG_SIZE = 640                # image size used for training
BATCH = 16                    # use 8 if you get an out-of-memory error
MAX_HUMAN_ONLY_IMAGES = 600   # limit for photos that contain only humans
NUM_BACKGROUND_IMAGES = 100   # photos without any human/animal (helps to avoid false detections)
SEED = 42

# ---------- Folders ----------
DRIVE_DIR = Path("/content/drive/MyDrive/human_animal_yolo")   # the trained model is saved here
COCO_DIR = Path("/content/coco_download")                      # raw COCO files (temporary)
DATASET_DIR = Path("/content/human_animal_dataset_v2")         # YOLO dataset (temporary)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
COCO_DIR.mkdir(parents=True, exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

In [ ]:
# Step 2: Download the COCO dataset (about 1 GB, takes 2-5 minutes)
# Photos: COCO val2017 (5,000 photos)    Labels: COCO 2017 annotations
!wget -c -q --show-progress -P {COCO_DIR} http://images.cocodataset.org/zips/val2017.zip
!wget -c -q --show-progress -P {COCO_DIR} http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# Unzip the photos and only the one label file we need
!unzip -q -n {COCO_DIR}/val2017.zip -d {COCO_DIR}
!unzip -q -n {COCO_DIR}/annotations_trainval2017.zip annotations/instances_val2017.json -d {COCO_DIR}

print("Download finished")
print("Photos found:", len(list((COCO_DIR / "val2017").glob("*.jpg"))))

In [ ]:
# Step 3: Build the YOLO dataset from the COCO files
# We keep the ORIGINAL 80 COCO class ids, so the pretrained model keeps everything it already knows.
# Only humans (COCO calls them "person") and 10 animals get labels.
from ultralytics import YOLO

ANIMALS = ["dog", "cat", "horse", "cow", "sheep", "elephant", "bear", "zebra", "giraffe", "bird"]
KEEP_NAMES = ["person"] + ANIMALS

base = YOLO(BASE_MODEL)                              # downloads the pretrained weights the first time
CLASS_NAMES = [base.names[i] for i in range(len(base.names))]
name_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
KEEP_IDS = [name_to_id[n] for n in KEEP_NAMES]
PERSON_ID = name_to_id["person"]
print("The model knows", len(CLASS_NAMES), "classes. We label these:", dict(zip(KEEP_NAMES, KEEP_IDS)))

with open(COCO_DIR / "annotations" / "instances_val2017.json") as f:
    coco = json.load(f)

coco_to_model_id = {c["id"]: name_to_id[c["name"]] for c in coco["categories"] if c["name"] in name_to_id}
image_info = {im["id"]: im for im in coco["images"]}

# YOLO label = class_id x_center y_center width height (all values between 0 and 1)
boxes = {}
for ann in coco["annotations"]:
    class_id = coco_to_model_id.get(ann["category_id"])
    if class_id not in KEEP_IDS or ann.get("iscrowd", 0) == 1:
        continue
    im = image_info[ann["image_id"]]
    W, H = im["width"], im["height"]
    x, y, w, h = ann["bbox"]                       # COCO box: left, top, width, height in pixels
    x1, y1 = max(x, 0), max(y, 0)
    x2, y2 = min(x + w, W), min(y + h, H)
    if x2 - x1 < 2 or y2 - y1 < 2:                 # skip tiny boxes
        continue
    boxes.setdefault(ann["image_id"], []).append(
        (class_id, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H)
    )

# Pick the photos: all photos with animals + some human-only photos + some background photos
random.seed(SEED)
with_animals = [i for i, b in boxes.items() if any(box[0] != PERSON_ID for box in b)]
human_only = [i for i, b in boxes.items() if all(box[0] == PERSON_ID for box in b)]
background = [i for i in image_info if i not in boxes]
random.shuffle(human_only)
random.shuffle(background)

chosen = with_animals + human_only[:MAX_HUMAN_ONLY_IMAGES] + background[:NUM_BACKGROUND_IMAGES]
random.shuffle(chosen)

# 80% train, 20% validation (the validation photos are never used for training)
n_val = int(len(chosen) * 0.2)
split_of = {image_id: ("val" if k < n_val else "train") for k, image_id in enumerate(chosen)}

# Create the folders and write the photos + label files
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ("train", "val"):
    (DATASET_DIR / split / "images").mkdir(parents=True)
    (DATASET_DIR / split / "labels").mkdir(parents=True)

class_count = {split: {class_id: 0 for class_id in KEEP_IDS} for split in ("train", "val")}
for image_id, split in split_of.items():
    file_name = image_info[image_id]["file_name"]
    shutil.copy(COCO_DIR / "val2017" / file_name, DATASET_DIR / split / "images" / file_name)
    lines = []
    for class_id, xc, yc, bw, bh in boxes.get(image_id, []):
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        class_count[split][class_id] += 1
    (DATASET_DIR / split / "labels" / (Path(file_name).stem + ".txt")).write_text("\n".join(lines))

# data.yaml tells YOLO where the data is and what the classes are
data_yaml = f"""path: {DATASET_DIR}
train: train/images
val: val/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
(DATASET_DIR / "data.yaml").write_text(data_yaml)

# Summary
print("Total photos:", len(chosen))
for split in ("train", "val"):
    n_images = sum(1 for s in split_of.values() if s == split)
    print(f"\n{split}: {n_images} photos")
    for class_id in KEEP_IDS:
        print(f"  {CLASS_NAMES[class_id]}: {class_count[split][class_id]} boxes")

In [ ]:
# Step 4: Check the ORIGINAL model on our validation photos (before any training)
before = base.val(data=str(DATASET_DIR / "data.yaml"), imgsz=IMG_SIZE, batch=BATCH, plots=False, verbose=False)

print()
print(f"BEFORE training -> Precision {before.box.mp:.3f} | Recall {before.box.mr:.3f} | "
      f"mAP50 {before.box.map50:.3f} | mAP50-95 {before.box.map:.3f}")

In [ ]:
# Step 5: Train (fine-tune) the model - about 5-10 minutes on a T4 GPU
model = YOLO(BASE_MODEL)

model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    seed=SEED,
    project=str(DRIVE_DIR / "runs"),     # training results are saved on Google Drive
    name="human_animal_finetune",
    exist_ok=True,
)

In [ ]:
# Step 6: Save the best model to Google Drive and compare BEFORE and AFTER training
candidates = list((DRIVE_DIR / "runs").rglob("human_animal_finetune/weights/best.pt"))
if not candidates:
    raise FileNotFoundError("best.pt was not found. Did Step 5 finish without errors?")
best_weights = max(candidates, key=lambda p: p.stat().st_mtime)

MODEL_FILE = DRIVE_DIR / "best_human_animal_v2.pt"
shutil.copy(best_weights, MODEL_FILE)
print("Best model found:", best_weights)
print("Copy saved to   :", MODEL_FILE)

best_model = YOLO(str(MODEL_FILE))
after = best_model.val(data=str(DATASET_DIR / "data.yaml"), imgsz=IMG_SIZE, batch=BATCH, plots=False, verbose=False)

print()
print("                  Precision  Recall   mAP50  mAP50-95")
print(f"BEFORE training   {before.box.mp:9.3f} {before.box.mr:7.3f} {before.box.map50:7.3f} {before.box.map:9.3f}")
print(f"AFTER training    {after.box.mp:9.3f} {after.box.mr:7.3f} {after.box.map50:7.3f} {after.box.map:9.3f}")
print("(Precision = how many detections are correct, Recall = how many real objects were found,")
print(" mAP = overall accuracy, higher is better, maximum 1.0)")

# Show the training curves and a sample of validation predictions
from IPython.display import Image as ShowImage, display

run_dir = best_weights.parent.parent
for chart in ["results.png", "val_batch0_pred.jpg"]:
    if (run_dir / chart).exists():
        display(ShowImage(filename=str(run_dir / chart)))

In [ ]:
# LAB DAY - Cell 7: load the model and the helper functions
!pip install -q ultralytics

import io
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from ultralytics import YOLO

# Which model to use? Keep ONE of the two MODEL_PATH lines.
MODEL_PATH = "/content/drive/MyDrive/human_animal_yolo/best_human_animal_v2.pt"   # our trained model
# MODEL_PATH = "yolov8s.pt"                                                       # original model without our training

if MODEL_PATH.startswith("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')

model = YOLO(MODEL_PATH)

HUMAN_NAMES = {"person", "human"}
ANIMAL_NAMES = {"dog", "cat", "horse", "cow", "sheep", "elephant", "bear", "zebra", "giraffe", "bird"}
HUMAN_IDS = [i for i, n in model.names.items() if n in HUMAN_NAMES]
ANIMAL_IDS = [i for i, n in model.names.items() if n in ANIMAL_NAMES]
print("Model loaded:", MODEL_PATH)
print("Human classes :", [model.names[i] for i in HUMAN_IDS])
print("Animal classes:", [model.names[i] for i in ANIMAL_IDS])


def detect(photo, conf=0.25):
    """Return a list of (label, score, (x1, y1, x2, y2)) for the humans and animals in a photo.
    Humans and animals are searched in two separate passes, so an animal box can never delete a
    human box that overlaps it (for example a man hugging a dog)."""
    found = []
    for class_ids, one_label_per_object in ((HUMAN_IDS, False), (ANIMAL_IDS, True)):
        result = model.predict(photo, conf=conf, classes=class_ids, agnostic_nms=one_label_per_object, verbose=False)[0]
        for box, class_id, score in zip(result.boxes.xyxy.tolist(), result.boxes.cls.tolist(), result.boxes.conf.tolist()):
            name = model.names[int(class_id)]
            label = "human" if name in HUMAN_NAMES else name
            found.append((label, score, tuple(box)))
    return found


def summarize(found):
    """Make a short sentence like: human x1, dog x2"""
    if not found:
        return "nothing detected"
    counts = Counter(label for label, _, _ in found)
    parts = []
    if "human" in counts:
        parts.append(f"human x{counts.pop('human')}")
    parts += [f"{name} x{n}" for name, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))]
    return ", ".join(parts)


def show(photo, found, title):
    """Draw the boxes on the photo: blue = human, orange = animal."""
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(photo)
    for label, score, (x1, y1, x2, y2) in found:
        color = "#1f6feb" if label == "human" else "#f08c00"
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=3))
        inside_top = y1 < 0.05 * photo.height          # no space above the box, so write the label inside it
        ax.text(x1 + 2, y1 + 2 if inside_top else y1 - 2, f"{label} {score:.0%}",
                color="white", fontsize=11, fontweight="bold",
                va="top" if inside_top else "bottom",
                bbox=dict(facecolor=color, edgecolor="none", pad=2))
    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
# LAB DAY - Cell 8: upload the photos given by the teacher and test them
from google.colab import files

CONF = 0.25     # a box is shown only if the model is at least this sure (try 0.15 if something is missed)
DEBUG = True    # True = also print everything the model sees (set it to False on the lab day)

print("Click 'Choose files' and select the photos...")
uploaded = files.upload()

for file_name, file_bytes in uploaded.items():
    photo = Image.open(io.BytesIO(file_bytes))
    photo = ImageOps.exif_transpose(photo).convert("RGB")    # fixes phone photos that are saved sideways
    found = detect(photo, CONF)
    summary = summarize(found)
    details = ", ".join(f"{label} {score:.0%}" for label, score, _ in found)
    print(f"{file_name}: {summary}   [{details}]")
    if DEBUG:
        raw = model.predict(photo, conf=0.10, verbose=False)[0]
        seen = ", ".join(f"{model.names[int(c)]} {float(p):.0%}" for c, p in zip(raw.boxes.cls.tolist(), raw.boxes.conf.tolist()))
        print("   everything the model sees (10% or more):", seen if seen else "nothing")
    show(photo, found, f"{file_name}  ->  {summary}")

In [ ]:
# OPTIONAL - Cell 9: download a few sample photos and run the model on them
import urllib.request
from pathlib import Path

from PIL import Image, ImageOps

if "detect" not in globals():
    raise RuntimeError("Run Cell 7 first (it loads the model).")

BASE_URL = "https://raw.githubusercontent.com/pjreddie/darknet/master/data"
PHOTOS = [
    "person.jpg",    # a person, a dog and a horse
    "dog.jpg",       # a dog (plus a bicycle and a truck, which are not shown)
    "horses.jpg",    # several horses
    "giraffe.jpg",   # a giraffe and a zebra
    "eagle.jpg",     # a bird
    "kite.jpg",      # small people on a beach (a harder photo)
]

TEST_DIR = Path("/content/test_photos")
TEST_DIR.mkdir(exist_ok=True)

for name in PHOTOS:
    if not (TEST_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", TEST_DIR / name)
print("Sample photos ready in", TEST_DIR)

for name in PHOTOS:
    photo = Image.open(TEST_DIR / name)
    photo = ImageOps.exif_transpose(photo).convert("RGB")
    found = detect(photo, 0.25)
    summary = summarize(found)
    print(f"{name}: {summary}")
    show(photo, found, f"{name}  ->  {summary}")

In [ ]:
# OPTIONAL - Cell 10: test the model on 100+ more photos that come with the correct answers
import os
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

from PIL import Image, ImageOps

if "detect" not in globals():
    raise RuntimeError("Run Cell 7 first (it loads the model).")
if len(model.names) != 80:
    raise RuntimeError("This test needs the 80-class model: best_human_animal_v2.pt or yolov8s.pt.")

CONF = 0.25           # same meaning as in Cell 8
SHOW_HOW_MANY = 9     # how many photos to draw (ALL photos are still checked and counted)
MIN_SIZE = 0.005      # objects smaller than 0.5% of the photo are ignored (too tiny to judge)

# 128 COCO photos with the correct answers (the first 128 photos of the COCO 2017 training set)
ZIP_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
WORK_DIR = Path("/content/coco128_test")
WORK_DIR.mkdir(exist_ok=True)
ZIP_FILE = WORK_DIR / "coco128.zip"
if not zipfile.is_zipfile(ZIP_FILE):              # missing, or only half downloaded because the internet was cut
    print("Downloading 128 photos (about 7 MB)...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
with zipfile.ZipFile(ZIP_FILE) as zf:
    zf.extractall(WORK_DIR)


def read_truth(photo_path):
    """Read the correct answer of a photo from its label file -> Counter like {'human': 2, 'horse': 1}."""
    label_path = Path(str(photo_path).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
    truth = Counter()
    if not label_path.exists():
        return truth
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5 or float(parts[3]) * float(parts[4]) < MIN_SIZE:
            continue
        name = model.names[int(parts[0])]
        if name in HUMAN_NAMES:
            truth["human"] += 1
        elif name in ANIMAL_NAMES:
            truth[name] += 1
    return truth


def kinds_seen(found, photo):
    """The labels the model found in a photo (tiny boxes are ignored, like in read_truth)."""
    area = photo.width * photo.height
    return {label for label, _, (x1, y1, x2, y2) in found if (x2 - x1) * (y2 - y1) >= MIN_SIZE * area}


def has_human(kinds):
    return "human" in kinds


def has_animal(kinds):
    return any(kind != "human" for kind in kinds)


def as_found(counts):
    """Write a Counter in the same format as detect(), so that summarize() can print it."""
    return [(label, 1.0, None) for label, n in counts.items() for _ in range(n)]


print("Checking the photos (about a minute)...")
results = []    # (photo path, correct answer, what the model found, labels the model found)
for photo_path in sorted(WORK_DIR.rglob("*.jpg")):
    truth = read_truth(photo_path)
    if not truth:
        continue                                  # no human and none of our 10 animals in this photo
    photo = ImageOps.exif_transpose(Image.open(photo_path)).convert("RGB")
    found = detect(photo, CONF)
    results.append((photo_path, truth, found, kinds_seen(found, photo)))

total = len(results)
if total == 0:
    raise RuntimeError(f"No photos with labels were found in {WORK_DIR}")
present_ok = sum(1 for _, truth, _, seen in results
                 if has_human(seen) == has_human(truth) and has_animal(seen) == has_animal(truth))
names_ok = sum(1 for _, truth, _, seen in results if seen == set(truth))

print(f"\nChecked {total} photos that contain a human or one of our 10 animals.")
print(f"Human / animal found or not found correctly : {present_ok} of {total} ({present_ok / total:.0%})")
print(f"Exact animal names also correct             : {names_ok} of {total} ({names_ok / total:.0%})")
print("Note: our training did not use these photos, but the original YOLOv8 saw them in its first training,")
print("so the score on brand new photos (the teacher's photos) can be a little lower.")

problems = [r for r in results if r[3] != set(r[1])]
if problems:
    print("\nPhotos to look at (truth = the correct answer):")
    for photo_path, truth, found, _ in problems[:12]:
        print(f"  {photo_path.name}: truth [{summarize(as_found(truth))}]   model [{summarize(found)}]")

# Draw some photos: human + animal first, then animal only, then human only, one after another
both = [r for r in results if has_human(r[1]) and has_animal(r[1])]
animal_only = [r for r in results if not has_human(r[1])]
human_only = [r for r in results if not has_animal(r[1])]
chosen, groups = [], [both, animal_only, human_only]
while len(chosen) < SHOW_HOW_MANY and any(groups):
    for group in groups:
        if group and len(chosen) < SHOW_HOW_MANY:
            chosen.append(group.pop(0))

for photo_path, truth, found, seen in chosen:
    photo = ImageOps.exif_transpose(Image.open(photo_path)).convert("RGB")
    verdict = "OK" if seen == set(truth) else "LOOK"
    show(photo, found, f"{photo_path.name}   [{verdict}]\ntruth: {summarize(as_found(truth))}   |   model: {summarize(found)}")

## Notes (for the viva)
- **Transfer learning**: YOLOv8 was already trained by Ultralytics on about 118,000 COCO photos. We start from those weights and fine-tune them on our human + animal photos.
- **Why we kept the 80-class list**: when the class list is changed, the class-recognition layers of the model are thrown away and must be learned again from our small dataset. Keeping the list keeps all the pretrained knowledge, so dogs, cows and humans are not mixed up.
- **Dataset**: COCO 2017 val2017 (5,000 photos). Only humans (COCO name: "person") and 10 animals are labeled. Labels are converted to YOLO format: `class x_center y_center width height`, all values between 0 and 1. 80% of the photos are used for training and 20% for validation.
- **Precision**: of all detections, how many are correct. **Recall**: of all real objects, how many were found.
- **mAP50 / mAP50-95**: the main accuracy score for detection (higher is better, maximum 1.0). Cell 6 prints the score BEFORE and AFTER training.
- **Confidence (CONF)**: a box is shown only if the model is at least this sure. Lower it to find more objects, raise it to remove wrong boxes.
- **Extra test (Cell 10)**: runs the model on 128 COCO photos with known answers and counts how often it finds the right human/animal names. The original YOLOv8 saw these photos in its first training, so brand new photos can score a little lower.